# 1. LOAD DATA

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')


file_path = './ai-impact-jobs-layoff-risk-dataset.csv'
df = pd.read_csv(file_path)

print("=" * 60)
print("Loading data")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df.head()}")

Loading data
Shape: (20000, 16)
Columns: ['Age', 'Education_Level', 'Years_of_Experience', 'Industry', 'Job_Role', 'Company_Size', 'Job_Level', 'Routine_Task_Percentage', 'Creativity_Requirement', 'Human_Interaction_Level', 'AI_Adoption_Level', 'Number_of_AI_Tools_Used', 'AI_Usage_Hours_Per_Week', 'Tasks_Automated_Percentage', 'AI_Training_Hours', 'Layoff_Risk']

First 5 rows:
   Age Education_Level  Years_of_Experience       Industry  \
0   59        Master's                    6        Finance   
1   44        Master's                   14  Manufacturing   
2   36      Bachelor's                    7         Retail   
3   27      Bachelor's                    6        Finance   
4   49     High School                   12        Finance   

                Job_Role Company_Size Job_Level  Routine_Task_Percentage  \
0             Accountant       Medium     Entry                       84   
1  Production Supervisor        Small     Entry                       30   
2          Store Ma

# 2. CHECK NULL VALUES

In [2]:
print("\n" + "=" * 60)
print("Checking null values")
print("=" * 60)
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if any(null_counts > 0) else "No null values found!")
print(f"Total null values: {df.isnull().sum().sum()}")



Checking null values
No null values found!
Total null values: 0


# 3. CHECK DUPLICATES

In [3]:
print("\n" + "=" * 60)
print("Checking duplicate records")
print("=" * 60)
duplicates = df.duplicated().sum()
print(f"Total duplicate records found: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Shape after removing duplicates: {df.shape}")


Checking duplicate records
Total duplicate records found: 0


# 4. CHECK DATA CONSISTENCY

In [4]:
print("\n" + "=" * 60)
print("Checking data consistency")
print("=" * 60)

# Age validation
invalid_age = df[(df['Age'] < 18) | (df['Age'] > 100)]
print(f"Invalid age (>100 or <18) {len(invalid_age)}")

# Years of experience validation (cannot be > age)
invalid_exp = df[df['Years_of_Experience'] > df['Age'] - 15]
print(f"Invalide experience (More than resoanble age) {len(invalid_exp)}")

# Percentages validation (should be 0-100)
percentage_cols = ['Routine_Task_Percentage', 'Creativity_Requirement', 
                    'Human_Interaction_Level', 'Tasks_Automated_Percentage']
for col in percentage_cols:
    invalid_pct = df[(df[col] < 0) | (df[col] > 100)]
    if len(invalid_pct) > 0:
        print(f"{col}: {len(invalid_pct)} invalid vlaue")


Checking data consistency
Invalid age (>100 or <18) 0
Invalide experience (More than resoanble age) 0


# 5. CHECK OUTLIERS (using IQR method)

In [5]:
print("\n" + "=" * 60)
print("Detecting outliers")
print("=" * 60)

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary[col] = len(outliers)
    
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outlier")
        print(f"  Range: {lower_bound:.2f} - {upper_bound:.2f}")

# OPTION: Cap outliers (Winsorization) - UNCOMMENT IF NEEDED
# for col in numeric_cols:
#     Q1 = df[col].quantile(0.25)
#     Q3 = df[col].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR
#     df[col] = df[col].clip(lower_bound, upper_bound)



Detecting outliers
Years_of_Experience: 46 outlier
  Range: -6.50 - 21.50
Number_of_AI_Tools_Used: 466 outlier
  Range: -3.50 - 8.50
AI_Usage_Hours_Per_Week: 721 outlier
  Range: -10.00 - 22.00
AI_Training_Hours: 949 outlier
  Range: -17.00 - 39.00


# 6. TARGET VARIABLE DISTRIBUTION

In [6]:
print("\n" + "=" * 60)
print("Checking tartget varable distribution")
print("=" * 60)
risk_dist = df['Layoff_Risk'].value_counts()
print(risk_dist)
print(f"\nPercentages: \n{df['Layoff_Risk'].value_counts(normalize=True) * 100}")


Checking tartget varable distribution
Layoff_Risk
High      6797
Low       6602
Medium    6601
Name: count, dtype: int64

Percentages: 
Layoff_Risk
High      33.985
Low       33.010
Medium    33.005
Name: proportion, dtype: float64


# 7. ENCODING CATEGORICAL VARIABLES

In [7]:
print("\n" + "=" * 60)
print("Encoding categorical variables")
print("=" * 60)

# 7.1 Ordinal Encoding (ordered categories)
ordinal_cols = ['Education_Level', 'Company_Size', 'Job_Level', 'AI_Adoption_Level']

# Define the correct order
education_order = [['High School', 'Bachelor\'s', 'Master\'s', 'PhD']]
company_size_order = [['Small', 'Medium', 'Large']]
job_level_order = [['Entry', 'Mid', 'Senior']]
ai_adoption_order = [['Low', 'Medium', 'High']]

# Apply ordinal encoder
ord_enc_edu = OrdinalEncoder(categories=education_order)
df['Education_Level'] = ord_enc_edu.fit_transform(df[['Education_Level']])

ord_enc_csize = OrdinalEncoder(categories=company_size_order)
df['Company_Size'] = ord_enc_csize.fit_transform(df[['Company_Size']])

ord_enc_jlevel = OrdinalEncoder(categories=job_level_order)
df['Job_Level'] = ord_enc_jlevel.fit_transform(df[['Job_Level']])

ord_enc_ai = OrdinalEncoder(categories=ai_adoption_order)
df['AI_Adoption_Level'] = ord_enc_ai.fit_transform(df[['AI_Adoption_Level']])

print("✓ Ordinal encoding completed")

# 7.2 Target encoding for Layoff_Risk
risk_order = [['Low', 'Medium', 'High']]
ord_enc_risk = OrdinalEncoder(categories=risk_order)
df['Layoff_Risk'] = ord_enc_risk.fit_transform(df[['Layoff_Risk']])
print("✓ Target encoding completed")

# 7.3 One-Hot Encoding for nominal columns
print("One-Hot Encoding for Industry and Job_Role...")

# Check unique values before encoding
print(f"  Unique Industries: {df['Industry'].nunique()}")
print(f"  Unique Job Roles: {df['Job_Role'].nunique()}")

# OPTION: Keep only top 20 Job Roles (reduce dimensionality)
# Uncomment if needed:
# top_roles = df['Job_Role'].value_counts().head(20).index
# df['Job_Role'] = df['Job_Role'].apply(lambda x: x if x in top_roles else 'Other')

# One-Hot Encoding
df = pd.get_dummies(df, columns=['Industry', 'Job_Role'], drop_first=False)
print(f"✓ One-Hot encoding completed. New shape: {df.shape}")



Encoding categorical variables
✓ Ordinal encoding completed
✓ Target encoding completed
One-Hot Encoding for Industry and Job_Role...
  Unique Industries: 8
  Unique Job Roles: 24
✓ One-Hot encoding completed. New shape: (20000, 46)


# 8. FEATURE SCALING (Min-Max Normalization)

In [9]:
print("\n" + "=" * 60)
print("Normalizing features (Min-Max Scaling)")
print("=" * 60)

# Separate features and target
X = df.drop('Layoff_Risk', axis=1)
y = df['Layoff_Risk']

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"X_scaled shape: {X_scaled.shape}")
print(f"Min value: {X_scaled.min().min():.4f}")
print(f"Max value: {X_scaled.max().max():.4f}")
print("✓ Scaling completed")



Normalizing features (Min-Max Scaling)
X_scaled shape: (20000, 45)
Min value: 0.0000
Max value: 1.0000
✓ Scaling completed


# 9. CORRELATION CHECK (Optional)

In [11]:
print("\n" + "=" * 60)
print("Checking collelation")
print("=" * 60)

# Calculate correlation with target
correlations = X_scaled.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 features with the highest correlation with the target:")
print(correlations.head(10))

# Check for highly correlated features (multicollinearity)
corr_matrix = X_scaled.corr()
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], 
                             corr_matrix.iloc[i, j]))

if high_corr:
    print(f"\n Features with high correlation (>0.8):")
    for pair in high_corr[:5]:
        print(f"  {pair[0]} - {pair[1]}: {pair[2]:.3f}")
else:
    print("No colleration found more than 0.8")



Checking collelation
Top 10 features with the highest correlation with the target:
Routine_Task_Percentage       0.781005
Creativity_Requirement        0.753955
Tasks_Automated_Percentage    0.747548
AI_Adoption_Level             0.602918
AI_Usage_Hours_Per_Week       0.504493
Number_of_AI_Tools_Used       0.499763
AI_Training_Hours             0.431510
Job_Level                     0.282160
Education_Level               0.171795
Human_Interaction_Level       0.159863
dtype: float64

 Features with high correlation (>0.8):
  Routine_Task_Percentage - Creativity_Requirement: -0.928
  Routine_Task_Percentage - Tasks_Automated_Percentage: 0.890
  Creativity_Requirement - Tasks_Automated_Percentage: -0.827
  AI_Adoption_Level - Number_of_AI_Tools_Used: 0.859
  AI_Adoption_Level - AI_Usage_Hours_Per_Week: 0.888


# 10. SAVE PREPROCESSED DATA

In [13]:
print("\n" + "=" * 60)
print("Saving preprocessed data:")
print("=" * 60)

# Save datasets
X_scaled.to_csv('X_scaled_features.csv', index=False)
y.to_csv('y_target.csv', index=False)

# Also save encoders and scaler for future use
import joblib
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(ord_enc_edu, 'ord_enc_edu.pkl')
joblib.dump(ord_enc_csize, 'ord_enc_csize.pkl')
joblib.dump(ord_enc_jlevel, 'ord_enc_jlevel.pkl')
joblib.dump(ord_enc_ai, 'ord_enc_ai.pkl')
joblib.dump(ord_enc_risk, 'ord_enc_risk.pkl')

print("Saved files:")
print("  - X_scaled_features.csv (Features)")
print("  - y_target.csv (Target)")
print("  - encoder and scaler.pkl files (For further use)")


Saving preprocessed data:
Saved files:
  - X_scaled_features.csv (Features)
  - y_target.csv (Target)
  - encoder and scaler.pkl files (For further use)


# 11. FINAL SUMMARY

In [15]:
print("\n" + "=" * 60)
print("Final preprocessing summery")
print("=" * 60)
print(f"Original shape: 20000 × 16")
print(f"Final shape: {X_scaled.shape[0]} × {X_scaled.shape[1]}")
print(f"Features after preprocessing: {X_scaled.shape[1]}")


Final preprocessing summery
Original shape: 20000 × 16
Final shape: 20000 × 45
Features after preprocessing: 45
